In [102]:
# Initial setup

In [103]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.3f}".format

# rng = np.random.RandomState(24)

In [104]:
# Data quality inspection

In [105]:
df = pd.read_csv("./data/raw/generation_tech_2015.csv")

In [106]:
df.head()

,MTU (CET/CEST),Area,Production Type,Generation (MW)
0,01/01/2015 00:00:00 - 01/01/2015 01:00:00,BZN|FR,Biomass,193.00
1,01/01/2015 01:00:00 - 01/01/2015 02:00:00,BZN|FR,Biomass,193.00
2,01/01/2015 02:00:00 - 01/01/2015 03:00:00,BZN|FR,Biomass,193.00
3,01/01/2015 03:00:00 - 01/01/2015 04:00:00,BZN|FR,Biomass,193.00
4,01/01/2015 04:00:00 - 01/01/2015 05:00:00,BZN|FR,Biomass,193.00


In [107]:
print(df.dtypes)
print("_____")
for column in df.columns:
    print(f'{column}: {df[column].apply(type).unique()}')

MTU (CET/CEST)     object
Area               object
Production Type    object
Generation (MW)    object
dtype: object
_____
MTU (CET/CEST): [<class 'str'>]
Area: [<class 'str'>]
Production Type: [<class 'str'>]
Generation (MW): [<class 'str'> <class 'float'>]


In [ ]:
# MTU is Market Time Unit. It's technically categorical data (hourly period).
# However, we'd like to easily extract the date of each MTU. We'll do this later.

# column "Generation (MW)" has a mix of data types, it should all be float. 
# If the value is non-numeric, then what are they instead?

In [109]:
s = df.copy()  # Make a temporary copy to avoid tampering with the original dataset
s['gen_float'] = pd.to_numeric(s['Generation (MW)'], errors='coerce')
s = s[s['gen_float'].isna()]
s

,MTU (CET/CEST),Area,Production Type,Generation (MW),gen_float
6080,11/09/2015 09:00:00 - 11/09/2015 10:00:00,BZN|FR,Biomass,NaN,NaN
7129,25/10/2015 02:00:00 (CEST) - 25/10/2015 02:00:00 (CET),BZN|FR,Biomass,NaN,NaN
7130,25/10/2015 02:00:00 (CET) - 25/10/2015 03:00:00,BZN|FR,Biomass,NaN,NaN
8760,01/01/2015 00:00:00 - 01/01/2015 01:00:00,BZN|FR,Energy storage,n/e,NaN
8761,01/01/2015 01:00:00 - 01/01/2015 02:00:00,BZN|FR,Energy storage,n/e,NaN
...,...,...,...,...,...
175199,31/12/2015 23:00:00 - 01/01/2016 00:00:00,BZN|FR,Wind Offshore,n/e,NaN
178360,12/05/2015 17:00:00 - 12/05/2015 18:00:00,BZN|FR,Wind Onshore,NaN,NaN
181280,11/09/2015 09:00:00 - 11/09/2015 10:00:00,BZN|FR,Wind Onshore,NaN,NaN
182329,25/10/2015 02:00:00 (CEST) - 25/10/2015 02:00:00 (CET),BZN|FR,Wind Onshore,NaN,NaN


In [110]:
s['Generation (MW)'].unique()

array([nan, 'n/e', ' '], dtype=object)

In [111]:
# Checking individual unique string value, we confirm that they all mean the same thing:
# there is no value. In which case, it's safe to coerce non-numerical values to NaN throughout the original dataset

In [112]:
df[df['Generation (MW)'] == 'n/e'].head()
df[df['Generation (MW)'] == ' '].head()
df[df['Generation (MW)'].isna()].head()

,MTU (CET/CEST),Area,Production Type,Generation (MW)
6080,11/09/2015 09:00:00 - 11/09/2015 10:00:00,BZN|FR,Biomass,NaN
7129,25/10/2015 02:00:00 (CEST) - 25/10/2015 02:00:00 (CET),BZN|FR,Biomass,NaN
7130,25/10/2015 02:00:00 (CET) - 25/10/2015 03:00:00,BZN|FR,Biomass,NaN
41120,11/09/2015 09:00:00 - 11/09/2015 10:00:00,BZN|FR,Fossil Gas,NaN
42169,25/10/2015 02:00:00 (CEST) - 25/10/2015 02:00:00 (CET),BZN|FR,Fossil Gas,NaN


In [113]:
df['Generation (MW)'] = pd.to_numeric(df['Generation (MW)'], errors='coerce')
for column in df.columns:
    print(f'{column}: {df[column].apply(type).unique()}')

MTU (CET/CEST): [<class 'str'>]
Area: [<class 'str'>]
Production Type: [<class 'str'>]
Generation (MW): [<class 'float'>]


In [ ]:
# MTU is Market Time Unit. It's technically categorical data (hourly period).
# However, we'd like to easily extract the date of each MTU.
# We'll use some text manipulation to extract the starting timestamp of each MTU.  
# The date will be a date 

In [115]:
df['start_time'] = df['MTU (CET/CEST)'].str.split(' - ').str[0]
df['date'] = df['start_time'].str.split(' ').str[0]
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

In [ ]:
# 

In [ ]:
 # Confirming that the dataset comes from the same Area / bidding zone
print(f'Same bidding zone: {df['Area'].unique()[0] == 'BZN|FR'}')

# And all technologies are consistently and concisely named
print(f'{df['Production Type'].unique()}')

Same bidding zone: True
['Biomass' 'Energy storage' 'Fossil Brown coal/Lignite'
 'Fossil Coal-derived gas' 'Fossil Gas' 'Fossil Hard coal' 'Fossil Oil'
 'Fossil Oil shale' 'Fossil Peat' 'Geothermal' 'Hydro Pumped Storage'
 'Hydro Run-of-river and pondage' 'Hydro Water Reservoir' 'Marine'
 'Nuclear' 'Other' 'Other renewable' 'Solar' 'Waste' 'Wind Offshore'
 'Wind Onshore']


In [126]:
# Remove irrelevant columns
df = df[['date','Production Type','Generation (MW)']]
df.head()

,date,Production Type,Generation (MW)
0,2015-01-01,Biomass,193.000
1,2015-01-01,Biomass,193.000
2,2015-01-01,Biomass,193.000
3,2015-01-01,Biomass,193.000
4,2015-01-01,Biomass,193.000


In [127]:
# Simplify column names to make the dataset easier to work with
# Keeping in mind that the standard unit for power in grid-scale is Megawatts (MW)

df = df.rename(columns={
    'Production Type':'tech',
    'Generation (MW)':'gen',
})

In [128]:
df

,date,tech,gen
0,2015-01-01,Biomass,193.000
1,2015-01-01,Biomass,193.000
2,2015-01-01,Biomass,193.000
3,2015-01-01,Biomass,193.000
4,2015-01-01,Biomass,193.000
...,...,...,...
183955,2015-12-31,Wind Onshore,"3,140.000"
183956,2015-12-31,Wind Onshore,"3,252.000"
183957,2015-12-31,Wind Onshore,"3,196.000"
183958,2015-12-31,Wind Onshore,"2,840.000"
